In [19]:
#라이브러리 불러오기
import joblib
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [ ]:
# Feature 데이터 불러오기
import joblib

feature_data = joblib.load("feature_data.pkl")

X = feature_data["X"]
y = feature_data["y"]
label_encoder = feature_data["label_encoder"]

print("X 크기:", X.shape)
print("y 크기:", y.shape)


X 크기: (9994, 668)
y 크기: (9994,)


In [ ]:
# 학습 데이터와 테스트 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("학습 데이터:", X_train.shape)
print("테스트 데이터:", X_test.shape)

print("\n학습 클래스")
print(pd.Series(y_train).value_counts().sort_index())

print("\n테스트 클래스")
print(pd.Series(y_test).value_counts().sort_index())

학습 데이터: (7995, 668)
테스트 데이터: (1999, 668)

학습 클래스
0     955
1    3481
2     278
3    3281
Name: count, dtype: int64

테스트 클래스
0    239
1    871
2     69
3    820
Name: count, dtype: int64


In [ ]:
# 모델 학습 함수 정의
def train_risk_model(
    X_train,
    y_train,
    model_type="random_forest",
):
    class_weights = {
        0: 5,  # CRITICAL
        1: 3,  # HIGH
        2: 1,  # LOW
        3: 3,  # MEDIUM
    }

    if model_type == "decision_tree":
        classifier = DecisionTreeClassifier(
            max_depth=20,
            min_samples_split=5,
            class_weight=class_weights,
            random_state=42,
        )

    elif model_type == "logistic_regression":
        classifier = LogisticRegression(
            max_iter=2000,
            class_weight=class_weights,
            solver="lbfgs",
            random_state=42,
        )

    elif model_type == "random_forest":
        classifier = RandomForestClassifier(
            n_estimators=300,
            max_depth=15,
            min_samples_split=5,
            min_samples_leaf=2,
            class_weight=class_weights,
            random_state=42,
            n_jobs=-1,
        )

    else:
        raise ValueError(
            "model_type은 decision_tree, "
            "logistic_regression, random_forest 중 하나여야 합니다."
        )

    classifier.fit(X_train, y_train)

    return classifier

In [ ]:
# 머신러닝 모델 학습
decision_tree_model = train_risk_model(
    X_train,
    y_train,
    model_type="decision_tree",
)

logistic_model = train_risk_model(
    X_train,
    y_train,
    model_type="logistic_regression",
)

random_forest_model = train_risk_model(
    X_train,
    y_train,
    model_type="random_forest",
)

print("모델 학습 완료")

모델 학습 완료


In [ ]:
# 학습 모델 저장
joblib.dump(
    decision_tree_model,
    "decision_tree_model.pkl"
)

joblib.dump(
    logistic_model,
    "logistic_model.pkl"
)

joblib.dump(
    random_forest_model,
    "risk_model.pkl"
)

joblib.dump(
    {
        "X_test": X_test,
        "y_test": y_test,
    },
    "test_data.pkl"
)

print("모델 및 테스트 데이터 저장 완료")

모델 및 테스트 데이터 저장 완료


<핵심 과정>
1. Feature 데이터 로드
2. Train/Test 분리
3. Decision Tree 학습
4. Logistic Regression 학습
5. Random Forest 학습
6. 모델 저장

<결과>
=> 동일한 Feature 데이터를 이용하여 Decision Tree, Logistic Regression, Random Forest 모델을 학습하였으며 그 학습된 모델을 저장하여 Evaluation 단계에서 Accuracy와 F1-score를 기준으로 성능을 비교할 수 있도록 하였다. 최종적으로 Random Forest가 가장 높은 Accuracy를 기록하여 최종 모델로 선정되었다.

<부연설명>
Model 단계에서는 동일한 데이터를 기반으로 세 가지 머신러닝 모델을 학습하고 저장하였으며, 이후 Evaluation 단계에서 성능을 비교하여 최종 모델을 선정하였습니다."